In [3]:
#### PACKAGES ####

import os
import cv2 as cv
import glob
import pandas as pd
import shutil
import random
from pathlib import Path


In [28]:
### INPUTS ###

INPUT_FOLDER = "/Volumes/research/LU24A1037-Jellyscope/Jellyscope/Monitoring data/Kristineberg_251128_sorted/ROIs"
OUTPUT_FOLDER = "/Volumes/research/LU24A1037-Jellyscope/Jellyscope/Training data/Monitoring_training_data/OG_training_data_2"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [29]:
# collect all label checker files

label_checker_files = glob.glob(os.path.join(INPUT_FOLDER, '**', 'LabelChecker_*.csv'), recursive=True)
print(f"Found {len(label_checker_files)} LabelChecker files.")

Found 871 LabelChecker files.


In [30]:
### Get label statistics across all LabelChecker files ###

label_stats = {}
for file_path in label_checker_files:

    df = pd.read_csv(file_path)
    counts = df["LabelTrue"].value_counts(dropna=True)

    for label, count in counts.items():
        label_stats[label] = label_stats.get(label, 0) + count

print(f"Label statistics across all LabelChecker ({len(label_checker_files)}) files: {len(label_stats)} labels found.")
for label, count in label_stats.items():
    print(f"{label}: {count}")


Label statistics across all LabelChecker (871) files: 17 labels found.
dentritus_glo: 730
dentritus: 577
filament_glo: 590
filament: 200
narcomedusa: 195
unknown: 115
copepod: 16
oikopleura: 163
mnemiopsis: 25
chaetognath: 34
pleurobrachia: 15
ctenophore_larvae: 40
ephyra_larvae: 2
hydromeduzae: 5
shrimp: 4
cyanea_larvae: 2
macro_filament: 1


In [ ]:
### Possibility to rename labels so it matches accross LabelChecker files and the training data folders
old_label = ""
new_label = ""

for file_path in label_checker_files:

    df = pd.read_csv(file_path)
    df["LabelTrue"] = df["LabelTrue"].replace(old_label, new_label)
    df.to_csv(file_path, index=False)

print(f"Renamed label '{old_label}' to '{new_label}' in all LabelChecker ({len(label_checker_files)}) files.")

Renamed label 'cyanea_larvae' to 'ephyra_larvae' in all LabelChecker (871) files.


In [25]:
### Sort images into folder stucture based on their label ###

#label_checker_files = glob.glob(os.path.join(INPUT_FOLDER, '**', 'LabelChecker_*.csv'), recursive=True)

for file_path in label_checker_files:
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)
    
    # collect labeled filenames and their labels
    # create a list where filneme is saved together with its label
    labeled_filenames = []
    for index, row in df.iterrows():
        label = row['LabelTrue']
        if pd.notna(label):
            filename = row['ImageFilename']
            labeled_filenames.append((filename, label))
    
    for filename, label in labeled_filenames:
        
        # if filename is already in the output folder, skip it
        # they might be replaced after QC, therefore path can be different, but filename is the same, 
        # so check if filename is already in the output folder
        already_exists = glob.glob(os.path.join(OUTPUT_FOLDER, '**', filename), recursive=True)
        
        if len(already_exists) > 0:
            continue
        
        # next to label checker csv file, there is a folder with images with the same name as the parent folder
        old_path = os.path.join(os.path.dirname(file_path), os.path.basename(os.path.dirname(file_path)), filename)
        new_path = os.path.join(OUTPUT_FOLDER, label, filename)
        
        # only skips if the it was not replaced after QC
        #if os.path.exists(new_path):
            #print(f"File {new_path} already exists, skipping...")
            #continue
        
        # copy file to new location, original location is preserved
        os.makedirs(os.path.dirname(new_path), exist_ok=True)
        shutil.copy2(old_path, new_path)
    
    print(f"Copied {len(labeled_filenames)} files")

Processing file: /Volumes/research/LU24A1037-Jellyscope/Jellyscope/Monitoring data/Kristineberg_251128_sorted/ROIs/q_batch_053/f_batch_053_06/LabelChecker_f_batch_053_06.csv
Copied 138 files
Processing file: /Volumes/research/LU24A1037-Jellyscope/Jellyscope/Monitoring data/Kristineberg_251128_sorted/ROIs/q_batch_053/f_batch_053_08/LabelChecker_f_batch_053_08.csv
Copied 185 files
Processing file: /Volumes/research/LU24A1037-Jellyscope/Jellyscope/Monitoring data/Kristineberg_251128_sorted/ROIs/q_batch_053/f_batch_053_01/LabelChecker_f_batch_053_01.csv
Copied 51 files
Processing file: /Volumes/research/LU24A1037-Jellyscope/Jellyscope/Monitoring data/Kristineberg_251128_sorted/ROIs/q_batch_053/f_batch_053_04/LabelChecker_f_batch_053_04.csv
Copied 4 files
Processing file: /Volumes/research/LU24A1037-Jellyscope/Jellyscope/Monitoring data/Kristineberg_251128_sorted/ROIs/q_batch_053/f_batch_053_03/LabelChecker_f_batch_053_03.csv
Copied 4 files
Processing file: /Volumes/research/LU24A1037-Jelly